# 00 — Environment and Data Audit

**Purpose:** Verify the research environment is reproducible and determine what data is available before any strategy work (mandate §8, §9).

**Research question:** Can this environment support the planned research — packages, LEAN access, data coverage, reproducibility?

**Data used:** none (environment introspection only).

**Methodology:** direct inspection of interpreter, packages, LEAN/QC availability, data directories, and config integrity; deterministic-seed check.

**Assumptions:** none.


In [1]:
import sys, platform, importlib, subprocess
print("python:", sys.version)
print("platform:", platform.platform())
for pkg in ["numpy", "pandas", "scipy", "statsmodels", "sklearn", "matplotlib", "yaml", "pytest"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:12s} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"{pkg:12s} MISSING")

python: 3.11.15 (main, Mar  3 2026, 09:26:23) [GCC 13.3.0]
platform: Linux-6.18.5-x86_64-with-glibc2.39
numpy        2.4.6


pandas       3.0.5
scipy        1.17.1


statsmodels  0.14.6


sklearn      1.9.0


matplotlib   3.11.1
yaml         6.0.1
pytest       9.1.1


In [2]:
import shutil, subprocess
print("lean CLI:", shutil.which("lean") or "NOT INSTALLED")
print("docker:", shutil.which("docker") or "NOT AVAILABLE")
try:
    import QuantConnect  # noqa
    print("QuantConnect Research: AVAILABLE")
except ImportError:
    print("QuantConnect Research: NOT AVAILABLE in this environment")

lean CLI: NOT INSTALLED
docker: /usr/bin/docker
QuantConnect Research: NOT AVAILABLE in this environment


In [3]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import yaml

RESEARCH_CONFIG = yaml.safe_load(open("../config/research_config.yaml"))
SEED = RESEARCH_CONFIG["meta"]["random_seed"]
np.random.seed(SEED)
print(f"config loaded | global seed = {SEED}")


config loaded | global seed = 20260801


In [4]:
# Contract-spec integrity: loads YAML and enforces tick_value == tick_size * multiplier
from spread_research.contract_metadata import load_contract_specs
specs = load_contract_specs("../config/instruments.yaml")
import pandas as pd
pd.DataFrame([vars(s) for s in specs.values()]).set_index("symbol")

,name,exchange,asset_class,multiplier,tick_size,tick_value,face_value
symbol,,,,,,,
MES,Micro E-mini S&P 500,CME,equity_index,5.0,0.250000,1.2500,NaN
MNQ,Micro E-mini Nasdaq-100,CME,equity_index,2.0,0.250000,0.5000,NaN
M2K,Micro E-mini Russell 2000,CME,equity_index,5.0,0.100000,0.5000,NaN
MYM,Micro E-mini Dow,CBOT,equity_index,0.5,1.000000,0.5000,NaN
ZT,2-Year US Treasury Note,CBOT,treasury,2000.0,0.003906,7.8125,200000.0
ZF,5-Year US Treasury Note,CBOT,treasury,1000.0,0.007812,7.8125,100000.0
ZN,10-Year US Treasury Note,CBOT,treasury,1000.0,0.015625,15.6250,100000.0
ZB,30-Year US Treasury Bond,CBOT,treasury,1000.0,0.031250,31.2500,100000.0


In [5]:
from pathlib import Path

DATA_DIR = Path("../data/processed")
DATA_AVAILABLE = any(DATA_DIR.glob("*_minute.*")) if DATA_DIR.exists() else False
if not DATA_AVAILABLE:
    print("BLOCKED-ON-DATA: no futures market data in this environment (see "
          "reports/00_repository_audit.md, issue L-001).\n"
          "Run this notebook inside QuantConnect Research, or drop licensed data\n"
          "into data/processed/ in the canonical schema (src/spread_research/data_loader.py).")


BLOCKED-ON-DATA: no futures market data in this environment (see reports/00_repository_audit.md, issue L-001).
Run this notebook inside QuantConnect Research, or drop licensed data
into data/processed/ in the canonical schema (src/spread_research/data_loader.py).


In [6]:
# Data inventory (mandate §8.1) — enumerates whatever exists in data/processed.
# Inside QuantConnect Research, replace this cell's source with QuantBook history
# calls per instrument and record: earliest/latest date, resolution, trade/quote/OI
# availability, session coverage, gaps (use spread_research.data_validation).
if DATA_AVAILABLE:
    from spread_research.data_loader import load_local
    from spread_research.data_validation import run_all_checks
    inventory = []
    for sym in specs:
        try:
            df = load_local(sym, "minute", DATA_DIR)
            issues = run_all_checks(df, sym)
            inventory.append({"symbol": sym, "start": df.index[0], "end": df.index[-1],
                              "bars": len(df), "issues": len(issues)})
        except FileNotFoundError:
            inventory.append({"symbol": sym, "start": None, "end": None, "bars": 0, "issues": None})
    display(pd.DataFrame(inventory))
else:
    print("Data inventory skipped — BLOCKED-ON-DATA (L-001)")

Data inventory skipped — BLOCKED-ON-DATA (L-001)


In [7]:
# Reproducibility check: the same seed must produce the same draw
import numpy as np
r1 = np.random.default_rng(SEED).normal(size=5)
r2 = np.random.default_rng(SEED).normal(size=5)
assert np.allclose(r1, r2)
print("deterministic RNG confirmed:", r1[:3])

deterministic RNG confirmed: [-0.66610039 -1.8138949  -1.32165393]


## Results

Executed in the initial (non-QC) research container — see cell outputs above. Summary:
scientific stack present and pinned; **no LEAN CLI, no QC access, no market data** (L-001);
contract-spec YAML passes internal consistency checks; RNG deterministic.

## Limitations

This audit reflects the setup container. It must be **re-run inside QuantConnect Research** to
complete the data-inventory section — that run is the actual verification of assumptions A-001/A-002.

## Decision

Framework development proceeds; all empirical work blocked pending data access (D-006).


## What this means for the algorithm

No algorithm-relevant conclusions yet. The environment can support the research once data access is provided. Nothing here validates any trading hypothesis.